<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/mva_2026_E06_01_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E06_01_Logistic_Regression ロジスティック回帰

- 学籍番号 [ ]
- 氏名 [ ]

- 切片 $\beta_0=$[ ]
- 回帰係数 $\beta_1=$[ ]
- Precision [ ]
- Recall [ ]
- F1-score [ ]


In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from sklearn.linear_model import LogisticRegression # scikit-learn のロジスティック回帰
from sklearn.model_selection import train_test_split # scikit-learn の訓練データとテストデータ分割ライブラリ
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score

%pip install japanize_matplotlib
import japanize_matplotlib # 日本語フォント

## データダウンロード

In [ ]:
df=pd.read_csv("https://www.data.math.ryukoku.ac.jp/course/mva_2026/data/classification.csv")
df

## 可視化して様子を見る

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['X'], bins=30, density=True, alpha=0.6, color='g', edgecolor='black')
plt.title('xのヒストグラム')
plt.xlabel('Value')
plt.ylabel('Density')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# 最小値と最大値から共通のビン（区切り）を定義する
bins = np.linspace(df['X'].min(), df['X'].max(), 30)

# データの分布を可視化 (さらにalphaを下げて透明度を調整)
plt.figure(figsize=(8, 5))
plt.hist(df[df['y']==0]['X'], bins=bins, density=False, alpha=0.20, color='r', edgecolor='black', label='y=0')
plt.hist(df[df['y']==1]['X'], bins=bins, density=False, alpha=0.08, color='b', edgecolor='black', label='y=1')
plt.hist(df['X'], bins=bins, density=False, alpha=0.08, color='g', edgecolor='black', label='全体')
plt.title('各群のヒストグラム')
plt.xlabel('x')
plt.ylabel('頻度')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
# 各コンポーネント（どのガウス分布から生成されたか）ごとにプロット
plt.scatter(df['X'], df['y'])

plt.title('散布図')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## ロジスティック回帰

In [ ]:
X=df.drop(['y'],axis=1) #説明変数
y=df['y'] #目的変数

# 訓練データ，テストデータ分割
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2
)

In [ ]:
# 1. 訓練データ (X_gmm_train, y_gmm_train) を用いてモデルを学習

# モデル
clf = LogisticRegression()
clf.fit(X_train, y_train)

# モデルのパラメータ（傾きと切片）を出力
print("--- ロジスティック回帰結果 ---")
print(f"回帰係数 (Slope): {clf.coef_[0][0]:.6f}")
print(f"切片 (Intercept): {clf.intercept_[0]:.6f}")

In [ ]:
## テストデータのみを対象とした予測確率と決定境界の可視化
import numpy as np
import matplotlib.pyplot as plt

# 評価対象をテストデータ (X_gmm_test, y_gmm_test) に限定

# グラフ描画用の等間隔な配列を作成
X_test_range = np.linspace(X_test.min() - 1, X_test.max() + 1, 1000).reshape(-1, 1)
y_prob_test = clf.predict_proba(X_test_range)[:, 1]

plt.figure(figsize=(10, 6))

# テストデータの散布図プロット
plt.scatter(X_test, y_test, c=y_test, cmap='coolwarm', alpha=0.9, edgecolors='black', linewidth=0.8, s=50, marker='s', label='Test Data (20%)')

# シグモイド曲線（予測確率）
plt.plot(X_test_range, y_prob_test, color='black', linewidth=2, label='Predicted Probability')

# 確率 0.5 となる決定境界（Decision Boundary）
decision_boundary = -clf.intercept_[0] / clf.coef_[0][0]
plt.axvline(x=decision_boundary, color='red', linestyle='--', label=f'Decision Boundary ({decision_boundary:.3f})')

# テスト精度を表示
plt.title(f'回帰曲線とテストデータ')
plt.xlabel('x')
plt.ylabel('Probability / Class Label')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 評価

#### 混同行列

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score, classification_report

y_test_pred = clf.predict(X_test)

# 2. 混同行列とF1スコアの算出
cm = confusion_matrix(y_test, y_test_pred)
print("混同行列:行（実測），列（予測）とも，0,1の順に並んでいることに注意")
print(cm)

print("\nClassification Report:(1の行が，1が陽性だと思った時のprecisionとrecall)")
print(classification_report(y_test, y_test_pred))

# これは accuracy だけを返す
accuracy = clf.score(X_test, y_test)
print(f"\n精度 (Accuracy): {accuracy:.4f}")

# これはf1スコア だけを返す
f1 = f1_score(y_test, y_test_pred, average='binary')
print(f"\nF1スコア (F1-score): {f1:.4f}")

#### ROC

In [ ]:
# テストデータに対する「陽性（クラス1）である予測確率」を取得
y_test_proba = clf.predict_proba(X_test)[:, 1]

# FPR (偽陽性率), TPR (真陽性率), しきい値を算出
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)

# AUCを計算
auc_score = roc_auc_score(y_test, y_test_proba)
print(f"AUC (Area Under Curve): {auc_score:.4f}")

# ROC曲線のプロット
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guess')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (偽陽性率)')
plt.ylabel('True Positive Rate (真陽性率)')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()